Import libraries

In [ ]:


import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns




Load the datasets

In [ ]:


students = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Students_Raw.csv')
courses = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Course_Catalogue.csv')
interactions = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Student_Course_Interactions.csv')
dictionary = pd.read_csv('C:\\Users\\vikes\\Favorites\\Links\\100daysinC\\explo_project\\Data_Dictionary.csv')




Create working copies

In [ ]:


students_raw = students.copy()
courses_raw = courses.copy()
interactions_raw = interactions.copy()
dictionary_raw = dictionary.copy()




Check initial Data dimensions

In [ ]:


datasets = {
    "Students": students,
    "Courses": courses,
    "Interactions": interactions
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isnull().sum().sum())




Students
Shape: (1020, 33)
Duplicate rows: 40
Missing values: 1455

Courses
Shape: (178, 20)
Duplicate rows: 0
Missing values: 288

Interactions
Shape: (5869, 10)
Duplicate rows: 114
Missing values: 5104

Handle Missing value placeholders

In [ ]:


missing_tokens = ["NA", "N/A", "null", "nan", "?", "Not Available", " ","na", "n/a", "Null", "NaN", "not available", "none", "None"]
def standardize_missing(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

        df[col] = df[col].mask(
            df[col].str.casefold().isin(missing_tokens), pd.NA
        )
    return df

students = standardize_missing(students_raw)
courses = standardize_missing(courses_raw)
interactions = standardize_missing(interactions_raw)
dictionary = standardize_missing(dictionary_raw)

print("Missing value placeholders standardized.")




Missing value placeholders standardized.

Remove exact duplicate rows

In [ ]:


print("Before removing duplicates:")

print("Students", students.duplicated().sum())
print("Courses", courses.duplicated().sum())
print("Interactions", interactions.duplicated().sum())

students = students.drop_duplicates()
courses = courses.drop_duplicates()
interactions = interactions.drop_duplicates()

print("After removing duplicates:")
print("Students", students.duplicated().sum())
print("Courses", courses.duplicated().sum())
print("Interactions", interactions.duplicated().sum())




Before removing duplicates:
Students 40
Courses 0
Interactions 114
After removing duplicates:
Students 0
Courses 0
Interactions 0

Standardize text formatting

In [ ]:


def clean_text(df):
    df = df.copy()

    for col in df.select_dtypes(include="object").columns:
        df[col] = (
            df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex= True)
        )
    return df

students = clean_text(students)
courses = clean_text(courses)
interactions = clean_text(interactions)

print("Text formatting standardized.")




Text formatting standardized.

Standardize important identifiers

In [ ]:


#Student IDs
students["Student_ID"] = students["Student_ID"].str.upper().str.strip()
interactions["Student_ID"] = interactions["Student_ID"].str.upper().str.strip()

# Course IDs
courses["Course_ID"] = courses["Course_ID"].str.upper().str.strip()
interactions["Course_ID"] = interactions["Course_ID"].str.upper().str.strip()

#Course codes
courses["Course Code"] = courses["Course Code"].str.upper().str.strip()
interactions["Course_Code"] = interactions["Course_Code"].str.upper().str.strip()
students["Selected_Course_Code"] = students["Selected_Course_Code"].str.upper().str.strip()

print("Identifier formatting completed.")




Identifier formatting completed.

Convert numerical columns

In [ ]:


student_numeric_cols = [
    "Age",
    "CGPA",
    "Previous_Semester_SGPA",
    "Attendance_Percentage",
    "Weekly_Study_Hours",
    "Num_Prior_Exploratory_Courses",
    "Num_Certifications",
    "Confusion_Level",
    "Selection_Confidence",
    "Previous_Course_Satisfaction"
]

for col in student_numeric_cols:
    students[col] = pd.to_numeric(
        students[col], errors="coerce"
    )

interaction_numeric_cols = [
    "Time_Spent_Minutes",
    "Rating_Given"
]

for col in interaction_numeric_cols:
    interactions[col] = pd.to_numeric(
        interactions[col], errors="coerce"
    )
course_numeric_cols = [
    "AY", "CREDIT", "Difficulty_Level", "Popularity_Index"
]

for col in course_numeric_cols:
    courses[col] = pd.to_numeric(
        courses[col], errors="coerce"
    )

print("Numerical data types converted.")

# Validate interaction time
invalid_time = interactions["Time_Spent_Minutes"] < 0
print("Invalid interaction time values:", invalid_time.sum())
interactions.loc[invalid_time, "Time_Spent_Minutes"] = np.nan




Numerical data types converted.
Invalid interaction time values: 20

Handle invalid numerical values

In [ ]:


student_ranges = {
    "Age": (17, 30),
    "CGPA": (0, 10),
    "Previous_Semester_SGPA": (0, 10),
    "Attendance_Percentage": (0, 100),
    "Weekly_Study_Hours": (0, 60),
    "Num_Prior_Exploratory_Courses": (0, 6),
    "Num_Certifications": (0, 8),
    "Confusion_Level": (1, 10),
    "Selection_Confidence": (1, 10),
    "Previous_Course_Satisfaction": (1, 5)
}

for col, (low, high) in student_ranges.items():
    invalid = (
        (students[col] < low) |
        (students[col] > high)
    )
    print(col, "invalid values:", invalid.sum())
    students.loc[invalid, col] = np.nan

#validate interaction ratings

invalid_rating = (
    (interactions["Rating_Given"] < 1) |
    (interactions["Rating_Given"] > 5)
)
print("Invalid ratings:", invalid_rating.sum())
interactions.loc[invalid_rating, "Rating_Given"] = np.nan

#validate course catalogue fields

for col in ["Difficulty_Level", "Popularity_Index"]:
    invalid = (courses[col] < 1) | (courses[col] > 5)
    print(col, "invalid values:", invalid.sum())
    courses.loc[invalid, col] = np.nan
invalid_credit = courses["CREDIT"] <= 0
print("Invalid credits:", invalid_credit.sum())
courses.loc[invalid_credit, "CREDIT"] = np.nan




Age invalid values: 16
CGPA invalid values: 13
Previous_Semester_SGPA invalid values: 4
Attendance_Percentage invalid values: 12
Weekly_Study_Hours invalid values: 14
Num_Prior_Exploratory_Courses invalid values: 10
Num_Certifications invalid values: 12
Confusion_Level invalid values: 17
Selection_Confidence invalid values: 13
Previous_Course_Satisfaction invalid values: 13
Invalid ratings: 11
Difficulty_Level invalid values: 0
Popularity_Index invalid values: 0
Invalid credits: 0

Standardize categorical labels

In [ ]:


student_category_cols = [
    "Gender",
    "Class12_Stream",
    "Skill_Level",
    "Learning_Preference",
    "Preferred_Difficulty",
    "Workload_Preference",
    "Preferred_Slot",
    "Internship_Experience"
]

for col in student_category_cols:
    students[col] = students[col].str.strip().str.lower()

interaction_category_cols = [
    "Interaction_Type",
    "Recommended_By",
    "Suitability_Label"
]

for col in interaction_category_cols:
    interactions[col] = interactions[col].str.strip().str.lower()

print("Selected categorical labels standardized.")




Selected categorical labels standardized.

Convert data columns

In [ ]:


students["Survey_Timestamp"] = pd.to_datetime(
    students["Survey_Timestamp"],
    format="mixed",
    errors="coerce"
)

interactions["Interaction_Date"] = pd.to_datetime(
    interactions["Interaction_Date"],
    format="mixed",
    errors="coerce"
)

print("Date conversion completed.")

print("Invalid student dates:",
      students["Survey_Timestamp"].isna().sum())

print("Invalid interaction dates:",
      interactions["Interaction_Date"].isna().sum())




Date conversion completed.
Invalid student dates: 62
Invalid interaction dates: 59

Invalid student dates: 62
Invalid interaction dates: 59

Handle missing numerical values

In [ ]:


for col in student_numeric_cols:
    median_value = students[col].median()
    if pd.notna(median_value):
        students[col] = students[col].fillna(median_value)

print("Student numerical missing values handled.")

#interaction time:

median_time = interactions["Time_Spent_Minutes"].median()
if pd.notna(median_time):
    interactions["Time_Spent_Minutes"] = (
        interactions["Time_Spent_Minutes"].fillna(median_time)
    )

print("Interaction time missing values handled.")




Student numerical missing values handled.
Interaction time missing values handled.

Inspect duplicate identifiers

In [ ]:


print("Duplicate Student IDs:",
      students["Student_ID"].duplicated().sum())

print("Duplicate Interaction IDs:",
      interactions["Interaction_ID"].duplicated().sum())

print("Duplicate Course IDs:",
      courses["Course_ID"].duplicated().sum())

student_duplicate_records = students[
    students["Student_ID"].duplicated(keep=False)
].sort_values(by="Student_ID")

interaction_duplicate_records = interactions[
    interactions["Interaction_ID"].duplicated(keep=False)
].sort_values(by="Interaction_ID")

course_duplicate_records = courses[
    courses["Course_ID"].duplicated(keep=False)
].sort_values(by="Course_ID")

print("Duplicate Student IDs:", students['Student_ID'].duplicated().sum())
print("Duplicate Interaction IDs:", interactions['Interaction_ID'].duplicated().sum())
print("Duplicate Course IDs:", courses['Course_ID'].duplicated().sum())

display(student_duplicate_records.head(10))
display(interaction_duplicate_records.head(10))
display(course_duplicate_records.head(10))




Duplicate Student IDs: 60
Duplicate Interaction IDs: 136
Duplicate Course IDs: 0
Duplicate Student IDs: 60
Duplicate Interaction IDs: 136
Duplicate Course IDs: 0
Student_ID
Batch_Year
Semester
Branch
School
Campus
Age
Gender
Class12_Stream
CGPA
...
Preferred_Difficulty
Workload_Preference
Preferred_Slot
Confusion_Level
Confusion_Reason
Selection_Confidence
Previous_Course_Satisfaction
Survey_Timestamp
Selected_Course_Code
Selected_Exploratory_Course
16
STU0015
2024
5
B.Tech Applied Petroleum Engineering
SOAE
Bidholi
20.0
MALE
PCM
<NA>
...
Moderate
HIGH
No Preference
7.0
Too many options
4.0
3.0
23-09-26
LSHP3038
History of Modern India, 1857-1947
374
STU0015
2024
5
B.Tech Applied Petroleum Engineering
SOAE
Bidholi
20.0
MALE
PCM
<NA>
...
Moderate
HIGH
No Preference
7.0
Too many options
4.0
3.0
23-09-26
LSHP3038
History of Modern India, 1857-1947
235
STU0028
2024
5
<NA>
SOCS
bidholi
<NA>
Male
PCM
6.96
...
Easy
Low
Slot 2
8.0
Unclear course content
3.0
3.0
46319.05231481481
DIGB2003P
Intr

saving the records

In [ ]:


import os
os.makedirs("../Cleaned_Data", exist_ok=True)

student_duplicate_records.to_csv(
    "../Cleaned_Data/Student_Duplicate_ID_Review.csv",
    index=False
)

interaction_duplicate_records.to_csv(
    "../Cleaned_Data/Interaction_Duplicate_ID_Review.csv",
    index=False
)

course_duplicate_records.to_csv(
    "../Cleaned_Data/Course_Duplicate_ID_Review.csv",
    index=False
)




Check missing values after cleaning

In [ ]:


cleaned_datasets = {
    "Students": students,
    "Courses": courses,
    "Interactions": interactions
}

for name, df in cleaned_datasets.items():

    print(f"\n========== {name} ==========")

    print("Remaining missing values:",
          df.isnull().sum().sum())

    print("\nMissing values column-wise:")

    print(
        df.isnull().sum()[
            df.isnull().sum() > 0
        ]
    )




========== Students ==========
Remaining missing values: 1067

Missing values column-wise:
Batch_Year                      18
Branch                          17
School                          20
Campus                          11
Gender                          24
Class12_Stream                  41
Internship_Experience           74
Interests                       43
Secondary_Interests            189
Hobbies                         71
Technical_Skills                54
Skill_Level                     41
Career_Goal                     75
Learning_Preference             32
Preferred_Difficulty            44
Workload_Preference             50
Preferred_Slot                  64
Confusion_Reason                64
Survey_Timestamp                62
Selected_Course_Code            43
Selected_Exploratory_Course     30
dtype: int64

========== Courses ==========
Remaining missing values: 288

Missing values column-wise:
Remarks                   151
Technical_Prerequisite    137
dtype: int6

Validate relationships between datasets

In [ ]:


# Identify unmatched Student IDs
student_ids = set(students["Student_ID"].dropna())

unmatched_student_records = interactions[
    ~interactions["Student_ID"].isin(student_ids)
]

# Identify unmatched Course IDs
course_ids = set(courses["Course_ID"].dropna())

unmatched_course_records = interactions[
    ~interactions["Course_ID"].isin(course_ids)
]

# Identify unmatched Course Codes
catalogue_codes = set(courses["Course Code"].dropna())

unmatched_selected_codes = students[
    students["Selected_Course_Code"].notna() &
    ~students["Selected_Course_Code"].isin(catalogue_codes)
]

unmatched_interaction_codes = interactions[
    interactions["Course_Code"].notna() &
    ~interactions["Course_Code"].isin(catalogue_codes)
]

print("Unmatched Student ID records:",
      len(unmatched_student_records))

print("Unmatched Course ID records:",
      len(unmatched_course_records))

print("Unmatched selected course codes:",
      len(unmatched_selected_codes))

print("Unmatched interaction course codes:",
      len(unmatched_interaction_codes))

# Export records for manual review
unmatched_student_records.to_csv(
    "../Cleaned_Data/Unmatched_Student_Records.csv", index=False
)

unmatched_course_records.to_csv(
    "../Cleaned_Data/Unmatched_Course_Records.csv", index=False
)

unmatched_selected_codes.to_csv(
    "../Cleaned_Data/Unmatched_Selected_Codes.csv", index=False
)

unmatched_interaction_codes.to_csv(
    "../Cleaned_Data/Unmatched_Interaction_Codes.csv", index=False
)




Unmatched Student ID records: 249
Unmatched Course ID records: 0
Unmatched selected course codes: 36
Unmatched interaction course codes: 130

Compare raw and cleaned datasets

In [ ]:


comparison = []
raw_datasets = {
    "Students": students_raw,
    "Courses": courses_raw,
    "Interactions": interactions_raw
}
for name in raw_datasets:
    raw_df = raw_datasets[name]
    clean_df = cleaned_datasets[name]
    comparison.append({
        "Dataset": name,
        "Raw Rows": len(raw_df),
        "Cleaned Rows": len(clean_df),
        "Raw Columns": len(raw_df.columns),
        "Cleaned Columns": len(clean_df.columns),
        "Raw Missing Values": raw_df.isnull().sum().sum(),
        "Cleaned Missing Values": clean_df.isnull().sum().sum(),
        "Raw Duplicates": raw_df.duplicated().sum(),
        "Cleaned Duplicates": clean_df.duplicated().sum()
    })
comparison_df = pd.DataFrame(comparison)
display(comparison_df)




Dataset
Raw Rows
Cleaned Rows
Raw Columns
Cleaned Columns
Raw Missing Values
Cleaned Missing Values
Raw Duplicates
Cleaned Duplicates
0
Students
1020
980
33
33
1455
1067
40
0
1
Courses
178
178
20
20
288
288
0
0
2
Interactions
5869
5755
10
10
5104
5140
114
0

Export cleaned datasets

In [ ]:


import os
os.makedirs("../Cleaned_Data", exist_ok=True)
students.to_csv(
    "../Cleaned_Data/Students_Cleaned.csv",
    index=False
)
courses.to_csv(
    "../Cleaned_Data/Course_Catalogue_Cleaned.csv",
    index=False
)
interactions.to_csv(
    "../Cleaned_Data/Student_Course_Interactions_Cleaned.csv",
    index=False
)
comparison_df.to_csv(
    "../Cleaned_Data/Cleaning_Comparison.csv",
    index=False
)
print("All cleaned datasets saved successfully!")




All cleaned datasets saved successfully!

Final Verification

In [ ]:


print("FINAL DATASET DIMENSIONS")
for name, df in cleaned_datasets.items():
    print(f"{name}: {df.shape}")
print("\nExact duplicate rows after cleaning")
for name, df in cleaned_datasets.items():
    print(name, ":", df.duplicated().sum())




FINAL DATASET DIMENSIONS
Students: (980, 33)
Courses: (178, 20)
Interactions: (5755, 10)

Exact duplicate rows after cleaning
Students : 0
Courses : 0
Interactions : 0

In [ ]:


print("Student Dataset Summary")
display(students.describe(include="all").T)

print("Course dataset summary")
display(courses.describe(include="all").T)

print("Interaction dataset summary")
display(interactions.describe(include="all").T)




Student Dataset Summary
count
unique
top
freq
mean
min
25%
50%
75%
max
std
Student_ID
980
960
STU0234
2
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Batch_Year
962
11
2025
425
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Semester
980
23
3
381
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Branch
963
148
B.Tech CSE - Data Science
91
NaN
NaN
NaN
NaN
NaN
NaN
NaN
School
960
32
SOCS
241
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Campus
969
13
Bidholi
449
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Age
980.0
<NA>
<NA>
<NA>
20.264286
17.0
20.0
20.0
21.0
27.0
1.009935
Gender
956
10
male
580
NaN
NaN
NaN
NaN
NaN
NaN
NaN
Class12_Stream
939
17
pcm
480
NaN
NaN
NaN
NaN
NaN
NaN
NaN
CGPA
980.0
<NA>
<NA>
<NA>
7.533071
0.0
6.9775
7.59
8.14
10.0
1.071788
Previous_Semester_SGPA
980.0
<NA>
<NA>
<NA>
7.580194
2.57
6.9375
7.55
8.29
10.0
1.071569
Attendance_Percentage
980.0
<NA>
<NA>
<NA>
79.303163
0.0
75.0
80.0
86.0
100.0
12.813365
Weekly_Study_Hours
980.0
<NA>
<NA>
<NA>
13.913265
0.0
9.0
13.0
17.0
45.0
6.731357
Num_Prior_Exploratory_Courses
980.0
NaN
NaN
NaN
1.189796
0.0
0.0
1.0
2.0
6.0
1.

Final Quality check

In [ ]:


print("FINAL DATA QUALITY CHECK")

for name, df in cleaned_datasets.items():

    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Exact duplicate rows:", df.duplicated().sum())
    print("Total missing values:", df.isnull().sum().sum())

    print("\nMissing values by column:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

# Check invalid interaction time
print("\nNegative interaction times:",
      (interactions["Time_Spent_Minutes"] < 0).sum())

# Check invalid ratings
print("Invalid ratings:",
      ((interactions["Rating_Given"] < 1) |
       (interactions["Rating_Given"] > 5)).sum())




FINAL DATA QUALITY CHECK

Students
Shape: (980, 33)
Exact duplicate rows: 0
Total missing values: 1067

Missing values by column:
Batch_Year                      18
Branch                          17
School                          20
Campus                          11
Gender                          24
Class12_Stream                  41
Internship_Experience           74
Interests                       43
Secondary_Interests            189
Hobbies                         71
Technical_Skills                54
Skill_Level                     41
Career_Goal                     75
Learning_Preference             32
Preferred_Difficulty            44
Workload_Preference             50
Preferred_Slot                  64
Confusion_Reason                64
Survey_Timestamp                62
Selected_Course_Code            43
Selected_Exploratory_Course     30
dtype: int64

Courses
Shape: (178, 20)
Exact duplicate rows: 0
Total missing values: 288

Missing values by column:
Remarks            